<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# 🔤 Working with Local LLMs: Model Management & Inference

Welcome to the second step of building your **AI Research Assistant**! In notebook 00, you validated that all platform services are running. Now it's time to **explore and manage your local AI models** so they're ready for the RAG pipeline and multi-agent system.

## 🎯 What You'll Learn

In this notebook, you'll:
- ✅ **Discover available models** — see what's been downloaded to your cluster
- ✅ **Check GPU resources** — understand your hardware capacity
- ✅ **Load and unload models** — manage GPU memory efficiently
- ✅ **Run chat completions** — generate text with your local LLM
- ✅ **Generate embeddings** — create vectors for the RAG pipeline

## 💡 Why This Matters

Thinkube manages LLMs differently from cloud APIs:

| Cloud APIs | Thinkube |
|-----------|----------|
| Pay per token | Run on your own GPUs — free |
| Always available | Models must be loaded onto a GPU first |
| Fixed model list | You choose which open-source models to download |
| Data sent to servers | Everything stays on your network |

The `tk-llm` SDK handles the lifecycle: **download → load → serve → unload**. You control what runs and when.

## 🏗️ What We're Building

```mermaid
graph LR
    A[🤖 Your Application<br/>LangChain/AG2] --> B[🔤 LLM Gateway<br/>Single API]
    B --> C[💬 Chat Model<br/>Ollama/vLLM<br/>e.g. Qwen 3.5 4B]
    B --> D[🧮 Embeddings<br/>TEI Backend<br/>e.g. Qwen3 Embedding 0.6B]
    
    style A fill:#e1f5ff,color:#1a1a1a
    style B fill:#fff4e6,color:#1a1a1a
    style C fill:#f3e5f5,color:#1a1a1a
    style D fill:#e8f5e9,color:#1a1a1a
```

### 🔄 The Model Lifecycle

1. **Mirror** 📥 — Download a model from HuggingFace to your local MLflow registry (done via Control Panel)
2. **Discover** 🔍 — Use `tk-llm` to see what models are available
3. **Load** ⬆️ — Load a model onto a GPU node
4. **Serve** 🚀 — The gateway routes inference requests to the right backend
5. **Unload** ⬇️ — Free GPU memory when you're done

---

**Prerequisites**: 
- ✅ Completed `00-platform-validation.ipynb`
- ✅ At least one model mirrored via the Control Panel (AI Models page)

Let's explore your models!

In [1]:
import os
from IPython.display import display, HTML

# Helper functions for colored output
def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✅ {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">❌ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ️  {msg}</span>'))

def skip(msg):
    display(HTML(f'<span style="color: #95a5a6;">⊘ {msg}</span>'))

# Import tk-llm
from tk_llm import LLMClient, get_openai_client

llm = LLMClient()
success("tk-llm client initialized")

In [2]:
# Discover all models in the registry
models = llm.list_models()
print(f"Total models: {models.total}")
print(f"  Available (loaded): {models.available}")
print(f"  Deployable (ready to load): {models.deployable}")
print(f"  Installed backends: {', '.join(models.installed_backend_types)}")
print()

# Show chat models
chat_models = [m for m in models.models if m.task == "text-generation"]
print(f"Chat models ({len(chat_models)}):")
for m in chat_models:
    status = "🟢 loaded" if m.state.value == "available" else "⚪ ready"
    print(f"  {status} {m.id} — {m.size or '?'} — {', '.join(m.server_type)}")

# Show embedding models
embed_models = [m for m in models.models if m.task == "feature-extraction"]
print(f"\nEmbedding models ({len(embed_models)}):")
for m in embed_models:
    status = "🟢 loaded" if m.state.value == "available" else "⚪ ready"
    print(f"  {status} {m.id} — {m.size or '?'} — {', '.join(m.server_type)}")

Total models: 13
  Available (loaded): 2
  Deployable (ready to load): 11
  Installed backends: ollama, vllm, text-embeddings

Chat models (11):
  ⚪ ready openai/gpt-oss-20b — ~21GB — tensorrt-llm
  ⚪ ready nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4 — ~4GB — tensorrt-llm
  ⚪ ready Qwen/Qwen3.5-4B — ~9GB — vllm
  ⚪ ready Qwen/Qwen3.5-9B — ~19GB — vllm
  ⚪ ready Qwen/Qwen3.5-27B — ~54GB — vllm
  🟢 loaded unsloth/Qwen3.5-4B-GGUF — ~3GB — ollama
  ⚪ ready unsloth/Qwen3.5-9B-GGUF — ~5GB — ollama
  ⚪ ready unsloth/Qwen3.5-27B-GGUF — ~15GB — ollama
  ⚪ ready unsloth/gemma-4-26B-A4B-it-GGUF — ~15GB — ollama
  ⚪ ready unsloth/Qwen3.5-9B — ~19GB — unsloth
  ⚪ ready unsloth/gemma-4-26B-A4B — ~53GB — unsloth

Embedding models (2):
  ⚪ ready nomic-ai/nomic-embed-text-v1.5 — ~0GB — text-embeddings
  🟢 loaded Qwen/Qwen3-Embedding-0.6B — ~1GB — text-embeddings


---
## 🖥️ Step 1: Check GPU Resources

Before loading models, let's see what hardware is available. Each GPU node shows its total memory, available slots (with time-slicing), and current allocations.

**💡 Key concepts:**
- **Slots** = virtual GPU devices created by time-slicing (e.g., 1 physical GPU → 4 slots)
- **Available slots** = how many more models can be loaded on this node
- **Shared memory** = time-slicing is active, models share GPU memory

In [3]:
# Check GPU resources across all nodes
gpu = llm.gpu_status()

print(f"Total GPU memory: {gpu.total_memory_gb:.0f} GB")
print(f"Used: {gpu.used_memory_gb:.0f} GB")
print(f"Can accept new model: {'Yes' if gpu.can_accept_new_model else 'No'}")
print()

for node in gpu.nodes:
    print(f"📍 {node.name}")
    print(f"   GPU: {node.gpu_product}")
    print(f"   Memory: {node.real_available_gb:.0f} GB available of {node.total_memory_gb:.0f} GB total")
    print(f"   Slots: {node.available_slots}/{node.total_slots} free {'(time-slicing active)' if node.shared_memory else ''}")
    if node.allocations:
        for alloc in node.allocations:
            print(f"   └─ {alloc.model_id} ({alloc.estimated_memory_gb:.1f} GB)")
    print()

Total GPU memory: 170 GB
Used: 27 GB
Can accept new model: Yes

📍 tkamd1
   GPU: NVIDIA-GeForce-RTX-3090-SHARED
   Memory: 47 GB available of 48 GB total
   Slots: 4/4 free (time-slicing active)

📍 tkspark
   GPU: NVIDIA-GB10-SHARED
   Memory: 96 GB available of 122 GB total
   Slots: 3/4 free (time-slicing active)
   └─ unsloth/Qwen3.5-4B-GGUF (16.3 GB)



---
## ⬆️ Step 2: Load a Chat Model

If no chat model is currently loaded, let's load one. We'll pick the smallest available model to save GPU memory.

**How loading works:**
1. `tk-llm` tells the pod manager to create a Deployment on the target GPU node
2. The backend (Ollama, vLLM, etc.) starts and pulls the model from MLflow
3. Once healthy, the gateway starts routing requests to it
4. The model state transitions: `deployable` → `loading` → `available`

💡 **Tip**: Loading takes 1-5 minutes depending on model size and backend type.

In [4]:
# Check if a chat model is already loaded
chat_available = [m for m in llm.list_models(state="available").models if m.task == "text-generation"]

if chat_available:
    CHAT_MODEL = chat_available[0].id
    success(f"Chat model already loaded: {CHAT_MODEL}")
else:
    # Find the smallest deployable chat model
    chat_deployable = [m for m in llm.list_models(state="deployable").models if m.task == "text-generation"]
    chat_deployable.sort(key=lambda m: m.params_b or 999)
    
    if not chat_deployable:
        error("No chat models available. Mirror a model first via the Control Panel → AI Models.")
    else:
        target = chat_deployable[0]
        info(f"Loading smallest chat model: {target.id} ({target.size})")
        
        # Pick the node with the most available memory
        gpu = llm.gpu_status()
        best_node = max(gpu.nodes, key=lambda n: n.real_available_gb or 0)
        info(f"Target node: {best_node.name} ({best_node.real_available_gb:.0f} GB free)")
        
        # The gateway's default context is 8k tokens. The RAG answers in
        # notebook 02 and the multi-agent debate in notebook 03 carry several
        # thousand tokens of retrieved evidence and tool results per turn, so
        # ask for a larger window up front (capped at the model's real limit).
        result = llm.load_model(target.id, node=best_node.name, max_context_length=32768)
        CHAT_MODEL = target.id
        info(f"State: {result.state} — {result.message}")

In [5]:
# Similarly, check if an embedding model is loaded
embed_available = [m for m in llm.list_models(state="available").models if m.task == "feature-extraction"]

if embed_available:
    EMBED_MODEL = embed_available[0].id
    success(f"Embedding model already loaded: {EMBED_MODEL}")
else:
    embed_deployable = [m for m in llm.list_models(state="deployable").models if m.task == "feature-extraction"]
    embed_deployable.sort(key=lambda m: m.params_b or 999)
    
    if not embed_deployable:
        error("No embedding models available. Mirror one via the Control Panel → AI Models.")
    else:
        target = embed_deployable[0]
        info(f"Loading smallest embedding model: {target.id} ({target.size})")
        
        gpu = llm.gpu_status()
        best_node = max(gpu.nodes, key=lambda n: n.real_available_gb or 0)
        info(f"Target node: {best_node.name} ({best_node.real_available_gb:.0f} GB free)")
        
        result = llm.load_model(target.id, node=best_node.name)
        EMBED_MODEL = target.id
        info(f"State: {result.state} — {result.message}")

---
## ✅ Step 3: Verify Models Are Ready

Let's check the current state of all backends and confirm our models are serving.

**Backend types:**
- **Ollama** — Runs quantized GGUF models, fastest to load, lower memory
- **vLLM** — Runs full-precision models, best throughput for larger models
- **TensorRT-LLM** — NVIDIA-optimized inference, highest performance
- **TEI** — Text Embeddings Inference, specialized for embedding models

In [6]:
# Check backends
backends = llm.list_backends()
print(f"Active backends: {backends.total} ({backends.healthy} healthy)")
print()

for b in backends.backends:
    status_icon = "🟢" if b.status == "healthy" else "🔴"
    print(f"{status_icon} {b.name} ({b.type})")
    print(f"   URL: {b.url}")
    print(f"   Models: {', '.join(b.models) if b.models else 'none'}")
    print()

Active backends: 2 (2 healthy)

🟢 Ollama (tkspark) (ollama)
   URL: http://10.1.3.156:11434
   Models: qwen3.5-4b:latest

🟢 TEI (tkspark) (text-embeddings)
   URL: http://10.1.3.143:7860
   Models: Qwen/Qwen3-Embedding-0.6B



---
## 🧪 Step 4: Test Chat Completions

Now let's use the loaded model through the gateway. The `get_openai_client()` function returns a standard OpenAI client that routes through the Thinkube LLM Gateway — same API you'd use with OpenAI, GPT-4, or Claude.

**💡 Why this is powerful**: You can swap models without changing any application code. Switch from a 4B Qwen to a 27B Qwen by just changing the model name.

In [ ]:
# Test chat completion
client = get_openai_client()

try:
    # Reasoning models (Qwen3, DeepSeek-R1, ...) think before they answer, and
    # the thinking is billed as completion tokens. Leave enough budget for the
    # visible answer, or it gets cut off before it starts.
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful research assistant. Be concise."},
            {"role": "user", "content": "What is LoRA fine-tuning? Explain in 2 sentences."}
        ],
        max_tokens=1024
    )

    choice = response.choices[0]
    answer = (choice.message.content or "").strip()
    reasoning = getattr(choice.message, "reasoning_content", None) or getattr(choice.message, "reasoning", None)
    if not answer:
        raise RuntimeError(
            f"no visible answer (finish_reason={choice.finish_reason}) — "
            "the token budget was spent on reasoning; raise max_tokens"
        )
    success(f"Chat completion with {CHAT_MODEL}")
    info(f"Tokens: {response.usage.prompt_tokens} prompt + {response.usage.completion_tokens} completion")
    if reasoning:
        info(f"The model reasoned for {len(reasoning)} characters before answering")
    print(f"\n💬 Response:\n{answer}")

except Exception as e:
    error(f"Chat completion failed: {e}")


---
## 🧮 Step 5: Test Embeddings

Embeddings convert text into **high-dimensional vectors** that capture semantic meaning. Similar concepts produce similar vectors — this is the foundation of RAG.

**Example:**
- "machine learning" → `[0.23, -0.45, 0.12, ...]` (1024 dimensions)
- "deep learning" → `[0.25, -0.43, 0.14, ...]` ← Very similar!
- "cooking recipes" → `[-0.67, 0.89, -0.34, ...]` ← Very different!

In the Research Assistant (notebook 02), we'll use embeddings to:
- Convert paper text into searchable vectors
- Find relevant papers by semantic similarity
- Build the retrieval step of our RAG pipeline

In [8]:
# Test embeddings
import numpy as np

try:
    texts = [
        "LoRA enables efficient fine-tuning by learning low-rank weight updates",
        "QLoRA combines quantization with LoRA for even lower memory usage",
        "The weather forecast predicts rain for tomorrow afternoon"
    ]
    
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    
    dim = len(response.data[0].embedding)
    success(f"Embeddings with {EMBED_MODEL} — {dim} dimensions")
    
    # Show semantic similarity
    vecs = [np.array(d.embedding) for d in response.data]
    
    def cosine_sim(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
    print(f"\n📐 Semantic Similarity:")
    print(f"  LoRA ↔ QLoRA:    {cosine_sim(vecs[0], vecs[1]):.3f}  (related topics)")
    print(f"  LoRA ↔ Weather:  {cosine_sim(vecs[0], vecs[2]):.3f}  (unrelated)")
    print(f"  QLoRA ↔ Weather: {cosine_sim(vecs[1], vecs[2]):.3f}  (unrelated)")
    
except Exception as e:
    error(f"Embedding test failed: {e}")


📐 Semantic Similarity:
  LoRA ↔ QLoRA:    0.833  (related topics)
  LoRA ↔ Weather:  0.289  (unrelated)
  QLoRA ↔ Weather: 0.260  (unrelated)


---
## ⬇️ Step 6: Unloading Models (Optional)

When you're done, you can unload models to free GPU memory. This deletes the backend Deployment but keeps the model in MLflow — you can reload it instantly later.

**When to unload:**
- You need GPU memory for a different, larger model
- You're switching from development to training workloads
- You want to test a different model variant

**💡 Tip**: Don't unload between notebooks! Keep your models loaded for the RAG pipeline (notebook 02) and multi-agent system (notebook 03).

In [9]:
# Uncomment to unload models:

# llm.unload_model(CHAT_MODEL)
# info(f"Unloaded {CHAT_MODEL}")

# llm.unload_model(EMBED_MODEL)
# info(f"Unloaded {EMBED_MODEL}")

info("Models are still loaded — ready for notebooks 02 and 03")

---
## 🎓 Summary: Your Local AI Models Are Ready!

Congratulations! You've explored and configured your local AI models. Here's what you accomplished:

### ✅ What You Did

1. ✅ **Discovered** all models in your cluster — chat and embedding
2. ✅ **Checked GPU resources** — available memory and time-sliced slots
3. ✅ **Loaded models** onto GPUs for serving
4. ✅ **Tested chat** — generated text with your local LLM
5. ✅ **Tested embeddings** — created vectors and measured semantic similarity

### 🎯 Key Takeaways

**The `tk-llm` SDK** gives you full control over your AI infrastructure:

```python
from tk_llm import LLMClient, get_openai_client

# Management
llm = LLMClient()
llm.list_models()              # What's available?
llm.gpu_status()               # How's my GPU doing?
llm.load_model("model-id")     # Load onto GPU
llm.unload_model("model-id")   # Free GPU memory

# Inference — standard OpenAI API
client = get_openai_client()
client.chat.completions.create(...)   # Chat
client.embeddings.create(...)         # Embeddings
```

**The OpenAI-compatible API** means your code works with any framework:
- 📚 **LangChain** — chains, retrievers, agents
- 🤖 **AG2/AutoGen** — multi-agent systems
- 🦜 **CrewAI** — collaborative AI agents
- 🔍 **LlamaIndex** — data frameworks

### 🚀 Next Steps

**Continue to**: `02-langchain-rag.ipynb`

In the next notebook, you'll use these models to build a **complete RAG pipeline**:
- 📥 Fetch research papers from ArXiv
- ✂️ Chunk papers into searchable pieces
- 🧮 Generate embeddings with your embedding model
- 💾 Store vectors in Qdrant
- 🔍 Perform semantic search
- 💬 Answer questions using your chat model
- 📈 Track everything with Langfuse

---

💡 **Remember**: Keep your models loaded for the next notebooks!